In [3]:
import zipfile
import os

In [6]:
# Unzip the dataset
with zipfile.ZipFile('keratoconus.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset')

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# This identifies the correct folder path for training
# Usually, it looks like 'dataset/Independent Test Set/Independent Test Set'
data_path = 'dataset/Independent Test Set/Independent Test Set'
print("Folders found:", os.listdir(data_path))

Folders found: ['Suspect', 'Keratoconus', 'Normal']


In [20]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
import numpy as np
from sklearn.utils import class_weight


In [21]:
# 1. IMAGE PREPARATION
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,        # Zoom in to help ignore outer text/borders
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Calculate Class Weights to fix "Normal" bias
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
dict_weights = dict(enumerate(weights))

Found 840 images belonging to 3 classes.
Found 210 images belonging to 3 classes.
AI Labels Verified: {0: 'Keratoconus', 1: 'Normal', 2: 'Suspect'}


In [22]:
from sklearn.utils import class_weight
import numpy as np

In [ ]:
# 2. THE MODEL
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Start frozen

model = models.Sequential([
    base_model,
    layers.GlobalMaxPooling2D(), # Better for detecting peak steepness
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),        # Higher dropout to stop overfitting
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# 3. TRAINING

# Callback to prevent overfitting
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

# Phase 1: Warm up (Top layers only)
print("Starting Phase 1: Training top layers...")
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=dict_weights,
    callbacks=[early_stop]
)

# Phase 2: Fine-Tuning (Unfreeze deep layers)
print("Starting Phase 2: Fine-Tuning deep layers...")
base_model.trainable = True

# We unfreeze more layers now (last 50) for better detail recognition
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.00001), # Tiny learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    class_weight=dict_weights,
    callbacks=[early_stop]
)

In [25]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

In [ ]:


image_to_test = '/content/dataset/Independent Test Set/Independent Test Set/Keratoconus/case18/KCN_18_CT_A.jpg'

def predict_eye_issue(img_path):

    if not os.path.exists(img_path):
        print(f"❌ Error: File not found at {img_path}")
        return None, 0

    # 1. Load and Preprocess
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # 2. Prediction
    prediction = model.predict(img_array, verbose=0)

    # 3. Class Mapping (Matches your Training Generator)
    class_labels = {0: 'Keratoconus', 1: 'Normal', 2: 'Suspect'}

    predicted_class_index = np.argmax(prediction)
    result = class_labels[predicted_class_index]
    confidence = prediction[0][predicted_class_index] * 100

    # 4. Diagnostic Output
    print("-" * 30)
    print(f"DIAGNOSTIC REPORT")
    print("-" * 30)
    print(f"File:       {os.path.basename(img_path)}")
    print(f"Result:     {result}")
    print(f"Confidence: {confidence:.2f}%")
    print("-" * 30)

    if confidence < 70:
        print("⚠️ WARNING: Low confidence score. Manual review recommended.")

    # 5. Visual Display
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.title(f"{result} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()

    return result, confidence


predict_eye_issue(image_to_test)

In [28]:
# Save the model to the Colab files sidebar
model.save('keratoconus_model.h5')

# Download the file to your local computer
from google.colab import files
files.download('keratoconus_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Install the converter
!pip install tensorflowjs

# Convert the model to a folder named 'tfjs_model'
import tensorflowjs as tfjs
tfjs.converters.save_keras_model(model, 'tfjs_model')

# Zip and download it
import shutil
shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')